<a href="https://colab.research.google.com/github/deepshresthaa/A-Clustered-Graph-Based-Framework-for-Semantic-Research-Paper-Retrieval-and-Recommendation/blob/main/code/04_training_gcn_for_all_45_clusters.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install torch-geometric
!pip install optuna

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv

In [ ]:
from google.colab import drive
import pandas as pd

drive.mount('/content/drive')

file_path = '/content/drive/MyDrive/papers_clustered.parquet'

df = pd.read_parquet(file_path)

df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,id,title,category,summary,embedding,pca_embedding,cluster
0,cs-9308101v1,Dynamic Backtracking,Artificial Intelligence,Because of their occasional need to return to ...,"[0.016953887, 0.060273074, -0.03308823, 0.0273...","[-0.02339239791035652, -0.3895048499107361, -0...",16
1,cs-9308102v1,A Market-Oriented Programming Environment and ...,Artificial Intelligence,Market price systems constitute a well-underst...,"[0.005659196, 0.03871944, -0.05143565, -0.0469...","[0.033656246960163116, -0.5250285863876343, -0...",12
2,cs-9309101v1,An Empirical Analysis of Search in GSAT,Artificial Intelligence,We describe an extensive study of search in GS...,"[-0.0076389345, -0.011536828, -0.024900155, -0...","[0.1429884433746338, -0.45451390743255615, -0....",12
3,cs-9311101v1,The Difficulties of Learning Logic Programs wi...,Artificial Intelligence,As real logic programmers normally use cut (!)...,"[-0.0024742228, 0.03417379, -0.02816961, 0.026...","[0.2859783172607422, -0.3050631880760193, -0.1...",12
4,cs-9311102v1,Software Agents: Completing Patterns and Const...,Artificial Intelligence,To support the goal of allowing users to recor...,"[0.04028217, -0.041410014, -0.0010439544, -0.0...","[0.35329899191856384, -0.17816203832626343, -0...",28


In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
from sklearn.metrics.pairwise import cosine_similarity
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
import optuna
import os
from tqdm import tqdm

optuna.logging.set_verbosity(optuna.logging.WARNING)

# 1. Standalone Optimal Threshold Finder Function
def find_optimal_threshold(cluster_df, threshold_range=np.arange(0.35, 0.85, 0.05)):
    """
    Evaluates graph connectivity metrics across thresholds to find the ideal balance
    between community structure (clustering coefficient) and edge density.
    """
    embeddings = np.vstack(cluster_df["pca_embedding"].values).astype(np.float32)
    sim_matrix = cosine_similarity(embeddings)
    np.fill_diagonal(sim_matrix, 0)

    results = []
    n_nodes = len(cluster_df)

    for thresh in threshold_range:
        adj = (sim_matrix >= thresh).astype(int)
        G = nx.from_numpy_array(adj)
        num_edges = G.number_of_edges()
        avg_degree = (2 * num_edges) / n_nodes if n_nodes > 0 else 0

        # Calculate clustering coefficient for reasonable graph sizes
        if 0 < num_edges < 15000:
            clustering_coeff = nx.average_clustering(G)
        else:
            clustering_coeff = 0.0

        components = nx.number_connected_components(G)

        results.append({
            "threshold": round(thresh, 2),
            "edges": num_edges,
            "avg_degree": round(avg_degree, 2),
            "clustering_coeff": round(clustering_coeff, 4),
            "components": components
        })

    res_df = pd.DataFrame(results)

    # Selection rule: Pick threshold with target degree ~ 4-15 and max clustering coefficient
    valid_candidates = res_df[(res_df["avg_degree"] >= 3) & (res_df["avg_degree"] <= 20)]

    if not valid_candidates.empty:
        best_row = valid_candidates.loc[valid_candidates["clustering_coeff"].idxmax()]
        best_threshold = float(best_row["threshold"])
    else:
        # Fallback to threshold that gives degree closest to 8 if out of bounds
        res_df["degree_diff"] = (res_df["avg_degree"] - 8).abs()
        best_threshold = float(res_df.loc[res_df["degree_diff"].idxmin()]["threshold"])

    return best_threshold, res_df


# 2. Graph Construction Function using Given Threshold
def build_cluster_graph(cluster_df, threshold):
    embeddings = np.vstack(cluster_df["pca_embedding"].values).astype(np.float32)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    x = torch.tensor(embeddings, dtype=torch.float, device=device)
    x_norm = F.normalize(x, p=2, dim=1)

    sim_matrix = torch.mm(x_norm, x_norm.t())
    sim_matrix.fill_diagonal_(0)

    src, dst = torch.where(sim_matrix >= threshold)
    edge_index = torch.stack([src, dst], dim=0)
    edge_weight = sim_matrix[src, dst]

    return Data(
        x=x.cpu(),
        edge_index=edge_index.cpu(),
        edge_attr=edge_weight.cpu(),
        paper_ids=cluster_df["id"].values,
        titles=cluster_df["title"].values,
        categories=cluster_df["category"].values,
        summaries=cluster_df["summary"].values
    )


# 3. Model Definition
class ClusterGraphSAGE(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, dropout=0.2):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, out_channels)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return F.normalize(x, p=2, dim=1)


# 4. Optuna Objective Builder (Using Fixed Graph per Cluster)
def make_objective(graph_data):
    def objective(trial):
        lr = trial.suggest_float("lr", 1e-3, 0.1, log=True)
        hidden_channels = trial.suggest_categorical("hidden_channels", [128, 256, 512])
        dropout = trial.suggest_float("dropout", 0.1, 0.5)

        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        model = ClusterGraphSAGE(
            in_channels=graph_data.num_features,
            hidden_channels=hidden_channels,
            out_channels=128,
            dropout=dropout
        ).to(device)

        data = graph_data.to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)

        model.train()
        epochs = 30
        final_loss = float('inf')

        if data.edge_index.size(1) == 0:
            return float('inf')

        for epoch in range(epochs):
            optimizer.zero_grad()
            out = model(data.x, data.edge_index)
            edge_src, edge_dst = data.edge_index[0], data.edge_index[1]
            pos_loss = -F.logsigmoid((out[edge_src] * out[edge_dst]).sum(dim=-1)).mean()
            neg_dst = torch.randint(0, data.num_nodes, (edge_src.size(0),), device=device)
            neg_loss = -F.logsigmoid(-(out[edge_src] * out[neg_dst]).sum(dim=-1)).mean()
            loss = pos_loss + neg_loss
            loss.backward()
            optimizer.step()
            final_loss = loss.item()

        return final_loss
    return objective

In [ ]:
import os
import gc
import torch
import torch.nn.functional as F
from tqdm import tqdm

output_dir = "/content/drive/MyDrive/gnn_cluster_models"
os.makedirs(output_dir, exist_ok=True)

unique_clusters = sorted(df["cluster"].unique())
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
best_thresholds_dict = {}

# Streamlined, memory-light threshold calculator
def find_optimal_threshold_light(cluster_df):
    embeddings = np.vstack(cluster_df["pca_embedding"].values).astype(np.float32)
    sim_matrix = cosine_similarity(embeddings)
    np.fill_diagonal(sim_matrix, 0)

    # Quick grid search across thresholds without creating heavy NetworkX graphs in RAM
    n_nodes = len(cluster_df)
    best_thresh = 0.65
    target_edges = n_nodes * 6  # target average degree ~12

    best_diff = float('inf')
    for thresh in np.arange(0.40, 0.85, 0.05):
        num_edges = np.sum(sim_matrix >= thresh) / 2  # undirected edges
        avg_deg = (2 * num_edges) / n_nodes if n_nodes > 0 else 0

        # We want average degree roughly between 6 and 15
        if 5 <= avg_deg <= 18:
            diff = abs(avg_deg - 10)
            if diff < best_diff:
                best_diff = diff
                best_thresh = float(thresh)

    del sim_matrix, embeddings
    return best_thresh

def process_cluster_safely(c_id):
    cluster_df = df[df["cluster"] == c_id].reset_index(drop=True)
    file_path = os.path.join(output_dir, f"cluster_{c_id}.pt")

    # 1. Lightweight optimal threshold calculation
    best_thresh = find_optimal_threshold_light(cluster_df)

    # 2. Build graph
    c_graph = build_cluster_graph(cluster_df, threshold=best_thresh)
    del cluster_df
    gc.collect()

    if c_graph.edge_index.size(1) < 5:
        torch.save({
            'model_state_dict': None,
            'best_params': None,
            'best_threshold': best_thresh,
            'embeddings': c_graph.x,
            'graph_data': c_graph
        }, file_path)
        del c_graph
        gc.collect()
        torch.cuda.empty_cache()
        return best_thresh

    # 3. Optuna HPO with ultra-low trial count to save RAM
    study = optuna.create_study(direction="minimize", storage=None)
    study.optimize(make_objective(c_graph), n_trials=5)  # Reduced to 5 trials
    best_params = study.best_params

    study.trials.clear()
    del study
    gc.collect()
    torch.cuda.empty_cache()

    # 4. Train full GNN model (fewer epochs to ensure safety)
    model = ClusterGraphSAGE(
        in_channels=c_graph.num_features,
        hidden_channels=best_params["hidden_channels"],
        out_channels=128,
        dropout=best_params["dropout"]
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=best_params["lr"])
    graph_data = c_graph.to(device)

    model.train()
    for epoch in range(100):  # Reduced to 100 epochs for high speed & zero memory bloat
        optimizer.zero_grad()
        out = model(graph_data.x, graph_data.edge_index)
        edge_src, edge_dst = graph_data.edge_index[0], graph_data.edge_index[1]
        pos_loss = -F.logsigmoid((out[edge_src] * out[edge_dst]).sum(dim=-1)).mean()
        neg_dst = torch.randint(0, graph_data.num_nodes, (edge_src.size(0),), device=device)
        neg_loss = -F.logsigmoid(-(out[edge_src] * out[neg_dst]).sum(dim=-1)).mean()
        loss = pos_loss + neg_loss
        loss.backward()
        optimizer.step()

    # 5. Extract embeddings and save checkpoint directly
    model.eval()
    with torch.no_grad():
        refined_embeddings = model(graph_data.x, graph_data.edge_index).cpu()

    torch.save({
        'model_state_dict': model.state_dict(),
        'best_params': best_params,
        'best_threshold': best_thresh,
        'embeddings': refined_embeddings,
        'graph_data': c_graph
    }, file_path)

    del model, optimizer, graph_data, c_graph, refined_embeddings
    gc.collect()
    torch.cuda.empty_cache()

    return best_thresh

# --- Ultra-Safe Chunked Loop (Batches of 2) ---
chunk_size = 2
for i in range(0, len(unique_clusters), chunk_size):
    cluster_chunk = unique_clusters[i:i + chunk_size]
    print(f"Processing Chunk: Clusters {cluster_chunk}")

    for c_id in cluster_chunk:
        best_thresh = process_cluster_safely(c_id)
        best_thresholds_dict[c_id] = best_thresh

    gc.collect()
    torch.cuda.empty_cache()

# Save final threshold mapping
torch.save(best_thresholds_dict, os.path.join(output_dir, "cluster_thresholds.pt"))


Processing Chunk: Clusters [np.int32(0), np.int32(1)]
Processing Chunk: Clusters [np.int32(2), np.int32(3)]
Processing Chunk: Clusters [np.int32(4), np.int32(5)]
Processing Chunk: Clusters [np.int32(6), np.int32(7)]
Processing Chunk: Clusters [np.int32(8), np.int32(9)]
Processing Chunk: Clusters [np.int32(10), np.int32(11)]
Processing Chunk: Clusters [np.int32(12), np.int32(13)]
Processing Chunk: Clusters [np.int32(14), np.int32(15)]
Processing Chunk: Clusters [np.int32(16), np.int32(17)]
Processing Chunk: Clusters [np.int32(18), np.int32(19)]
Processing Chunk: Clusters [np.int32(20), np.int32(21)]
Processing Chunk: Clusters [np.int32(22), np.int32(23)]
Processing Chunk: Clusters [np.int32(24), np.int32(25)]
Processing Chunk: Clusters [np.int32(26), np.int32(27)]
Processing Chunk: Clusters [np.int32(28), np.int32(29)]
Processing Chunk: Clusters [np.int32(30), np.int32(31)]
Processing Chunk: Clusters [np.int32(32), np.int32(33)]
Processing Chunk: Clusters [np.int32(34), np.int32(35)]
Pr

In [ ]:
import numpy as np
import torch
import networkx as nx
import os

# 1. Compute cluster center centroids from the main dataframe
cluster_ids = sorted(df["cluster"].unique())
cluster_centers_dict = {}

for c_id in cluster_ids:
    c_df = df[df["cluster"] == c_id]
    c_embeddings = np.vstack(c_df["pca_embedding"].values)
    cluster_centers_dict[c_id] = np.mean(c_embeddings, axis=0)

# Shape: (45, feature_dim)
cluster_centers_matrix = np.vstack([cluster_centers_dict[c] for c in cluster_ids])


# 2. Localized p-q graph search function
def multi_cluster_p_q_search(
    query_vector,
    cluster_centers_matrix=cluster_centers_matrix,
    p=1.0,
    q=0.5,
    max_depth=3,
    rel_decay_cutoff=0.15
):
    # Step A: Centroid routing (Find max similarity cluster)
    query_norm = query_vector / np.linalg.norm(query_vector)
    centers_norm = cluster_centers_matrix / np.linalg.norm(cluster_centers_matrix, axis=1, keepdims=True)
    cluster_sims = centers_norm @ query_norm
    target_cluster = int(np.argmax(cluster_sims))

    print(f"Routing to Cluster ID: {target_cluster} (Centroid Cosine Similarity: {cluster_sims[target_cluster]:.4f})")

    # Step B: Load target cluster's saved checkpoint from Google Drive
    checkpoint_path = f"/content/drive/MyDrive/gnn_cluster_models/cluster_{target_cluster}.pt"
    if not os.path.exists(checkpoint_path):
        raise FileNotFoundError(f"Checkpoint for cluster {target_cluster} not found at {checkpoint_path}")

    cluster_data = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    graph_data = cluster_data['graph_data']
    gnn_embeddings = cluster_data['embeddings'] # Trained GNN vectors

    # Step C: Localized search within target cluster graph using GNN embeddings
    # Safely convert both query and embeddings to CPU PyTorch tensors
    # Step C: Localized search using raw node features (graph_data.x) to match dimensions
    query_t = torch.as_tensor(query_vector, dtype=torch.float32).unsqueeze(0)

    # graph_data.x contains the original node features matching the input dimensions
    node_sims = torch.cosine_similarity(query_t, graph_data.x, dim=1).detach().numpy()
    seed_idx = int(np.argmax(node_sims))

    # Build NetworkX graph for traversal from saved PyG edge_index
    G = nx.Graph()
    edges = graph_data.edge_index.numpy()
    for u, v in zip(edges[0], edges[1]):
        G.add_edge(u, v)

    visited_scores = {}
    queue = [(seed_idx, 0, 1.0)]

    while queue:
        curr_node, depth, prob = queue.pop(0)
        if curr_node in visited_scores or depth > max_depth:
            continue

        combined_score = 0.6 * float(node_sims[curr_node]) + 0.4 * prob
        visited_scores[curr_node] = combined_score

        neighbors = list(G.neighbors(curr_node)) if G.has_node(curr_node) else []
        for nbr in neighbors:
            bias = (1 / p) if nbr == seed_idx else (1 / q)
            next_prob = prob * (1.0 / len(neighbors)) * bias
            queue.append((nbr, depth + 1, next_prob))

    ranked_nodes = sorted(visited_scores.items(), key=lambda x: x[1], reverse=True)

    # Step D: Adaptive Cutoff to pick top relevant papers
    final_selected_papers = []
    prev_score = ranked_nodes[0][1] if ranked_nodes else 0
    for idx, score in ranked_nodes:
        if len(final_selected_papers) >= 5 and (prev_score - score) > rel_decay_cutoff:
            break
        final_selected_papers.append({
            "paper_id": graph_data.paper_ids[idx],
            "title": graph_data.titles[idx],
            "category": graph_data.categories[idx],
            "score": round(score, 4)
        })
        prev_score = score

    return target_cluster, final_selected_papers


# 3. End-to-End Prediction Test Helper
def search_by_abstract_text(reference_pca_row_index=2):
    query_vector = df.loc[reference_pca_row_index, "pca_embedding"].astype(np.float32)
    sample_title = df.loc[reference_pca_row_index, "title"]
    actual_cluster = df.loc[reference_pca_row_index, "cluster"]

    print(f"Test Query Paper Title: '{sample_title}' (Ground-truth cluster: {actual_cluster})\n")

    cluster_id, results = multi_cluster_p_q_search(
        query_vector=query_vector,
        cluster_centers_matrix=cluster_centers_matrix
    )

    print(f"\nTop Recommended Papers from Cluster {cluster_id}:")
    for r in results:
        print(f" - [{r['score']}] {r['title']} (ID: {r['paper_id']})")

    return cluster_id, results

# Run the test
test_cluster, recommended_papers = search_by_abstract_text(reference_pca_row_index=2)

Test Query Paper Title: 'An Empirical Analysis of Search in GSAT' (Ground-truth cluster: 12)

Routing to Cluster ID: 12 (Centroid Cosine Similarity: 0.5600)

Top Recommended Papers from Cluster 12:
 - [1.0] An Empirical Analysis of Search in GSAT (ID: cs-9309101v1)
 - [0.9026] Backbone Fragility and the Local Search Cost Peak (ID: abs-1106.0240v1)
 - [0.8829] Evolving difficult SAT instances thanks to local search (ID: abs-1011.5866v1)
 - [0.7404] Symbiosis of Search and Heuristics for Random 3-SAT (ID: abs-1402.4455v1)
 - [0.549] On the accuracy and running time of GSAT (ID: cs-0002003v1)
 - [0.5245] A Logical Approach to Efficient Max-SAT solving (ID: cs-0611025v1)
 - [0.5188] A Linear Weight Transfer Rule for Local Search (ID: abs-2303.14894v1)
 - [0.5124] Decomposing Hard SAT Instances with Metaheuristic Optimization (ID: abs-2312.10436v1)
 - [0.5038] Community-based 3-SAT Formulas with a Predefined Solution (ID: abs-1902.09706v1)
 - [0.5028] Explaining SAT Solving Using Causal Rea